# Setup Inicial

## Importando Bibliotecas

In [1]:
import os
from google.cloud import bigquery

## Autenticando o Client do BigQuery

In [2]:
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = '../bigquery_api_key.json'
client = bigquery.Client()

# Questão 1 — Extração e Análise Exploratória com SQL

## 1.1 — Extração de Dados (SQL)

### Qual é o volume de sessões, usuários únicos e transações por mês?

In [ ]:
query_1 = """
SELECT
  DATE_TRUNC(PARSE_DATE('%Y%m%d', date), MONTH) as month_year,
  COUNT(DISTINCT visitId) AS unique_visits,
  COUNT(DISTINCT fullVisitorId) AS unique_users,
  SUM(totals.transactions) AS transactions
FROM
  `bigquery-public-data.google_analytics_sample.ga_sessions_*`
GROUP BY
  month_year
ORDER BY
  month_year
"""

unique_visits_df = client.query(query_1).to_dataframe()

display(unique_visits_df)

c:\Users\arthu\Documents\teste_etus\questao_1\venv\Lib\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,month_year,unique_visits,unique_users,transactions
0,2016-08-01,73494,61699,1241
1,2016-09-01,69729,59121,904
2,2016-10-01,95409,84901,919
3,2016-11-01,111201,99734,955
4,2016-12-01,77589,63839,1450
5,2017-01-01,63585,53041,713
6,2017-02-01,61122,51364,733
7,2017-03-01,68574,57888,993
8,2017-04-01,65935,55681,959
9,2017-05-01,64217,52233,1160


### Quais canais de aquisição (source/medium) geram mais receita? E quais têm melhor taxa de conversão?

In [60]:
query_2 = """
WITH top_6_revenue AS (
  SELECT
    CONCAT(trafficSource.source, ' / ', trafficSource.medium) AS acquisition_channel,
    SUM(COALESCE(totals.totalTransactionRevenue, 0)) AS revenue
  FROM
    `bigquery-public-data.google_analytics_sample.ga_sessions_*`
  GROUP BY
    acquisition_channel
  ORDER BY
    revenue DESC
  LIMIT 6
)

SELECT
  *,
  CONCAT(FORMAT('%.2f', (revenue / SUM(revenue) OVER ()) * 100), '%') AS revenue_share,
  CASE 
    WHEN acquisition_channel = '(direct) / (none)' THEN NULL 
    ELSE CONCAT(
           FORMAT(
             '%.2f', 
             (revenue / SUM(
                CASE WHEN acquisition_channel = '(direct) / (none)' THEN 0 ELSE revenue END
             ) OVER()) * 100
           ),
           '%'
         ) 
  END AS revenue_share_excluding_direct
FROM
  top_6_revenue
ORDER BY
  revenue DESC
"""

source_revenue = client.query(query_2).to_dataframe()

display(source_revenue)

c:\Users\arthu\Documents\teste_etus\questao_1\venv\Lib\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,acquisition_channel,revenue,revenue_share,revenue_share_excluding_direct
0,(direct) / (none),1332591980000,75.76%,None
1,google / organic,238974970000,13.59%,56.05%
2,dfa / cpm,128787150000,7.32%,30.21%
3,google / cpc,27876250000,1.58%,6.54%
4,mail.google.com / referral,24830780000,1.41%,5.82%
5,dealspotr.com / referral,5873640000,0.33%,1.38%


Acima, há o Top 6 canais que mais trazem receita para a empresa. Observa-se que o canal direto detém a grande maioria da receita. Excluindo-o para que se tenha uma noção melhor do share de receita entre os canais de marketing, observa-se que pesquisas no google orgânicas e "dfa/cpm" trazem quase 90% das demais receitas.

In [72]:
query_3 = """
SELECT
  CONCAT(trafficSource.source, ' / ', trafficSource.medium) AS acquisition_channel,
  COUNT(*) AS total_sessions,
  CONCAT(
    FORMAT(
      '%.2f', 
      SUM(COALESCE(totals.transactions, 0)) / COUNT(*) * 100
    ),
    '%'
  ) AS conversion_rate
FROM
  `bigquery-public-data.google_analytics_sample.ga_sessions_*`
GROUP BY
  acquisition_channel
ORDER BY
  SUM(COALESCE(totals.transactions, 0)) / COUNT(*) DESC
LIMIT 5
"""

convertion_rate_df = client.query(query_3).to_dataframe()

display(convertion_rate_df)

c:\Users\arthu\Documents\teste_etus\questao_1\venv\Lib\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,acquisition_channel,total_sessions,conversion_rate
0,basecamp.com / referral,2,50.00%
1,mail.aol.com / referral,3,33.33%
2,us-mg5.mail.yahoo.com / referral,4,25.00%
3,calendar.google.com / referral,5,20.00%
4,chat.google.com / referral,7,14.29%


Observa-se, que o Top 5 de canais com a maior taxa de conversão é bem diferente dos canais com maior receita. Entretanto, isso se deve ao fato de terem um baixo número de sessões, logo, é mais fácil terem uma taxa de conversão maior. 

Ao se remover os canais com baixo número de sessões obtem-se a seguinte tabela:

In [73]:
query_4 = """
SELECT
  CONCAT(trafficSource.source, ' / ', trafficSource.medium) AS acquisition_channel,
  COUNT(*) AS total_sessions,
  CONCAT(
    FORMAT(
      '%.2f', 
      SUM(COALESCE(totals.transactions, 0)) / COUNT(*) * 100
    ),
    '%'
  ) AS conversion_rate
FROM
  `bigquery-public-data.google_analytics_sample.ga_sessions_*`
GROUP BY
  acquisition_channel
HAVING
  total_sessions > 100
ORDER BY
  SUM(COALESCE(totals.transactions, 0)) / COUNT(*) DESC
LIMIT 5
"""

convertion_rate_df_filtered = client.query(query_4).to_dataframe()

display(convertion_rate_df_filtered)

c:\Users\arthu\Documents\teste_etus\questao_1\venv\Lib\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,acquisition_channel,total_sessions,conversion_rate
0,dealspotr.com / referral,528,7.58%
1,mail.google.com / referral,1457,4.80%
2,l.facebook.com / referral,795,3.77%
3,groups.google.com / referral,1025,3.71%
4,google / cpm,496,3.63%


Desta forma, conseguimos um dado mais confiável para qual é o canal com maior conversão. Neste caso, é o `dealspotr.com / referral` que também está presente nos top canais de receita, sendo o 6° colocado.